# Week 2 — GROUP BY, Aggregates, HAVING: Review Score Analysis
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Apply `GROUP BY` and aggregate functions to analyse the Olist review score distribution
- Turn a group count into a share of the whole using a scalar subquery in the `SELECT` list
- Explain why an average and the shape of a distribution can tell two different stories

On Wednesday you learned the machinery: `GROUP BY`, the five aggregates, and `HAVING`.
Today you point that machinery at one table — `order_reviews` — and use it to answer a
question a business actually argues about: *are our customers happy?*

### Setup — run this cell first

It loads the eight Olist tables into a SQLite database and connects the `%%sql` magic to
it. Nothing below will run until this cell has finished. You should see
`order_reviews: 99,224 rows` in the output — that is today's table.

In [1]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


Mounted at /content/drive
Searching your Google Drive for phase-2-python-sql.zip ...
Unzipping phase-2-python-sql.zip ...
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 497.5/497.5 kB 32.5 MB/s eta 0:00:00


## Why this matters

Every Olist order can be reviewed with a star rating from 1 to 5, and the
`order_reviews` table holds **99,224** of those reviews. Suppose the marketing team
wants a line for the investor deck. Someone runs a single query, gets an average score
of **4.09**, and writes *"our customers rate us 4.09 out of 5."* Technically true.

But here is the thing that should make you suspicious. **57.8%** of all those reviews
are a perfect 5 stars — well over half. If most customers are giving full marks, why
isn't the average sitting up around 4.6 or 4.7? Where did the missing half-star go?

That gap between "most reviews are 5-star" and "the average is 4.09" is not a rounding
error, and it is not a bug in the query. It is the single most important thing about
this dataset's reviews — and one `GROUP BY` is enough to expose it. Let's build the
query that explains it.

## 1. The distribution — one row per star rating

An average squeezes 99,224 numbers into one. A **distribution** keeps the shape: how
many reviews landed on each possible score. That is exactly the question `GROUP BY`
answers — bucket every row by its `review_score`, then `COUNT(*)` each bucket.

Notice how little SQL this takes. `review_score` only has five distinct values (1 to 5),
so we get exactly five rows back and the whole table becomes readable at a glance. We
`ORDER BY review_score` rather than by the count, because for a distribution you want
the natural 1→5 sequence, not the biggest bucket first — you are reading the *shape*,
not ranking anything.

In [2]:
%%sql
-- How many reviews landed on each star rating?
-- Expected (5 rows): 1 -> 11,424 | 2 -> 3,151 | 3 -> 8,179 | 4 -> 19,142 | 5 -> 57,328
--                    (these five counts sum to 99,224 — every review in the table)
SELECT review_score,
       COUNT(*) AS count
FROM order_reviews
GROUP BY review_score
ORDER BY review_score

,review_score,count
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


## 2. From counts to shares — the scalar subquery

`57,328` is a big number, but big compared to what? To turn a group count into a
percentage you divide it by the total row count of the whole table. The problem is that
`GROUP BY` has already split the table into buckets — inside the query, no bucket knows
how big the table was.

The fix is a **scalar subquery**: `(SELECT COUNT(*) FROM order_reviews)`. It is a
complete query in parentheses that returns exactly one value, so SQL can drop it into
an arithmetic expression as if you had typed `99224` yourself. It runs once, and every
group divides by the same number.

One detail is doing a lot of work here: the `100.0`, not `100`. In SQLite, dividing an
integer by an integer gives you an integer — the decimals are thrown away, not rounded.
Writing `100.0` makes one operand a REAL and forces real division. We'll look at exactly
what goes wrong without it in the Common Mistakes section.

In [3]:
%%sql
-- The review score distribution, with each bucket as a share of all 99,224 reviews.
-- Expected: 1 -> 11,424 (11.5%) | 2 -> 3,151 (3.2%) | 3 -> 8,179 (8.2%)
--           4 -> 19,142 (19.3%) | 5 -> 57,328 (57.8%)
SELECT review_score,
       COUNT(*) AS count,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM order_reviews), 1) AS percentage
FROM order_reviews
GROUP BY review_score
ORDER BY review_score

,review_score,count,percentage
0,1,11424,11.5
1,2,3151,3.2
2,3,8179,8.2
3,4,19142,19.3
4,5,57328,57.8


## 3. The overall average — one aggregate, no groups

An aggregate function does not *require* a `GROUP BY`. If you leave it out, SQL treats
the entire table as a single group and returns exactly one row. That is how you get a
headline number: `AVG(review_score)` over all 99,224 rows.

We wrap it in `ROUND(..., 2)` because the raw average is a long float, and we ask for
`MIN` and `MAX` in the same breath to confirm the scale really does run 1 to 5 — a cheap
sanity check that costs nothing and catches bad data. `COUNT(*)` alongside them tells us
how many rows the average was computed over, which is the first question any careful
reader will ask.

In [4]:
%%sql
-- The single headline number — plus the context needed to trust it.
-- Expected: total_reviews 99,224 | overall_avg 4.09 | lowest 1 | highest 5
SELECT COUNT(*)                     AS total_reviews,
       ROUND(AVG(review_score), 2)  AS overall_avg,
       MIN(review_score)            AS lowest,
       MAX(review_score)            AS highest
FROM order_reviews

,total_reviews,overall_avg,lowest,highest
0,99224,4.09,1,5


## 4. Reading the two results together

Now put the last two queries side by side, because together they answer the question we
opened with.

**57.8% of reviews are 5-star, yet the average is 4.09, not 5.0.** The reason is
visible in the distribution: the second-largest bucket is not 4-star, it is **1-star**,
at 11,424 reviews (11.5%). Add the 3.2% who gave 2 stars and roughly one review in seven
is a bottom-two score.

A mean is a balancing point, and low scores have a long lever. Moving a customer from
5 stars to 4 costs the average one point of that customer's contribution; moving them
from 5 to 1 costs four. So that 11.5% of angry customers drags the mean down about four
times harder per person than the 19.3% of merely-satisfied 4-star customers do. The
happy majority cannot pull it back up, because they have already hit the ceiling at 5 —
there is no 6 stars.

This is a **bimodal** distribution: people either love it or they hate it, with very
few in the middle (only 3.2% at 2 stars, 8.2% at 3). If that pattern feels familiar, it
should — it is the same shape you met in Phase 1 with the Amazon product reviews, and it
turns up almost everywhere customers rate things voluntarily. People with a strong
feeling in either direction are the ones who bother to write a review at all.

**The business takeaway:** reporting "4.09 average" hides the real story. The number
that should be on the dashboard is the 1-star count. Fixing the average means finding
out what happened to those 11,424 orders — not persuading 4-star customers to give 5.

## Going deeper — bringing `HAVING` back to the distribution

Wednesday's `HAVING` clause fits straight onto this query. Suppose an analyst only cares
about the score buckets that are large enough to be worth investigating, and sets the bar
at more than 10,000 reviews. That is a statement about a whole *group*, not about any
individual row — so it belongs in `HAVING`, after the aggregation, not in `WHERE`.

Watch what survives the filter. From the counts above: 11,424 and 19,142 and 57,328 clear
10,000; 3,151 and 8,179 do not. The 2-star and 3-star buckets disappear and we are left
with three rows — the two extremes plus the 4-star bucket. The percentages still read
against the full 99,224, because the scalar subquery is computed over the untouched table
and is not affected by `HAVING` at all. That is a useful property: your shares stay
honest even when you filter the display.

In [5]:
%%sql
-- Only the score buckets carrying more than 10,000 reviews.
-- 2-star (3,151) and 3-star (8,179) are filtered out by HAVING.
-- Expected (3 rows): 5 -> 57,328 (57.8%) | 4 -> 19,142 (19.3%) | 1 -> 11,424 (11.5%)
SELECT review_score,
       COUNT(*) AS count,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM order_reviews), 1) AS percentage
FROM order_reviews
GROUP BY review_score
HAVING COUNT(*) > 10000       -- filters GROUPS, after the counting
ORDER BY count DESC

,review_score,count,percentage
0,5,57328,57.8
1,4,19142,19.3
2,1,11424,11.5


## Common mistakes

**Mistake 1 — forgetting to force real division.** This is the trap in today's headline
query, and it is nasty because it does not raise an error. In SQLite `11424 / 99224` is
integer division: the answer is `0.115...`, the decimal part is *discarded*, and you get
`0`. A whole percentage column of zeros, no warning.

Worse is the near-miss. If you multiply by `100` first — `COUNT(*) * 100 / total` — the
numbers stop being zero and start looking plausible: `11`, `3`, `8`, `19`, `57`. They are
still wrong. Every one has been truncated, not rounded, so 11.5 became 11 and 57.8 became
57. Those five "percentages" now sum to 98, not 100, and nobody notices until someone
checks the totals in a meeting. Writing `100.0` — a REAL literal — fixes both versions.

**Mistake 2 — dividing by the group instead of the table.** If you write
`COUNT(*) * 100.0 / COUNT(*)` you get 100.0 on every row, because both `COUNT(*)`s are
evaluated inside the same group. The denominator must come from the scalar subquery, which
is computed over the whole table before any grouping happens.

In [6]:
%%sql
-- ── COMMON MISTAKE 1: integer division ──────────────────────────────
-- WRONG — plain integer division; every percentage truncates to 0:
--   SELECT review_score,
--          COUNT(*) / (SELECT COUNT(*) FROM order_reviews) AS pct
--   FROM order_reviews GROUP BY review_score
--   -> 1 -> 0 | 2 -> 0 | 3 -> 0 | 4 -> 0 | 5 -> 0
--
-- ALSO WRONG — * 100 (an INTEGER) looks plausible but is still truncated:
--   SELECT review_score,
--          COUNT(*) * 100 / (SELECT COUNT(*) FROM order_reviews) AS pct
--   FROM order_reviews GROUP BY review_score
--   -> 1 -> 11 | 2 -> 3 | 3 -> 8 | 4 -> 19 | 5 -> 57   (sums to 98, not 100)
--
-- CORRECT — 100.0 is a REAL, so the division keeps its decimals:
SELECT review_score,
       COUNT(*) AS count,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM order_reviews), 1) AS percentage
FROM order_reviews
GROUP BY review_score
ORDER BY review_score;         -- Expected: 11.5 | 3.2 | 8.2 | 19.3 | 57.8 (sums to 100.0)

,review_score,count,percentage
0,1,11424,11.5
1,2,3151,3.2
2,3,8179,8.2
3,4,19142,19.3
4,5,57328,57.8


In [7]:
%%sql
-- ── COMMON MISTAKE 2: the wrong denominator ─────────────────────────
-- WRONG — both COUNT(*) calls belong to the same group, so every row is 100.0:
--   SELECT review_score, ROUND(COUNT(*) * 100.0 / COUNT(*), 1) AS pct
--   FROM order_reviews GROUP BY review_score
-- CORRECT — the denominator is a scalar subquery over the WHOLE table.
-- Same idea, phrased as "what share of all reviews is this bucket?":
SELECT review_score,
       COUNT(*)                                    AS bucket_size,
       (SELECT COUNT(*) FROM order_reviews)        AS all_reviews,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM order_reviews), 1) AS percentage
FROM order_reviews
GROUP BY review_score
ORDER BY review_score          -- Expected: all_reviews is 99,224 on every row

,review_score,bucket_size,all_reviews,percentage
0,1,11424,99224,11.5
1,2,3151,99224,3.2
2,3,8179,99224,8.2
3,4,19142,99224,19.3
4,5,57328,99224,57.8


## Mini-challenge — your turn

⏱ ~5–10 minutes

The analytics lead wants a one-line answer: **how many review-score buckets contain more
than 5,000 reviews?** Not the counts themselves — just how many of the five score buckets
clear that bar.

**Expected:** **4** of the five buckets. Check it against the distribution you already
have: 5-star (57,328), 4-star (19,142), 1-star (11,424) and 3-star (8,179) all clear
5,000; only 2-star (3,151) falls short.

Two ways to get there, both fine:
1. Write the `GROUP BY` + `HAVING COUNT(*) > 5000` query and count the rows it returns
   by eye — this is Wednesday's `HAVING` pattern with a new threshold.
2. Wrap that query in parentheses as a subquery and count it in SQL:
   `SELECT COUNT(*) AS n FROM ( ...your grouped query... )`.

Try approach 1 first, then see if you can get approach 2 to work.

In [8]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT review_score, COUNT(*) AS review_count
FROM order_reviews
GROUP BY review_score
HAVING COUNT(*) > 5000
ORDER BY review_count DESC;

,review_score,review_count
0,5,57328
1,4,19142
2,1,11424
3,3,8179


## Group exercise — the rest of the session

For the remainder of today you'll work in your groups through the exercises notebook.
Five questions, all answerable with what you now know, and each one has a self-check cell
that prints a ✅ when your query is right:

1. What is the average number of items per order? (a `GROUP BY` inside a subquery — group
   `order_items` by `order_id` first, then average the counts)
2. Which customer state has the highest average review score? — a **deliberate trap**.
   Try it, then work out as a group *why* it cannot be written yet. The answer is the
   subject of Week 3.
3. How many payment records use more than 6 installments? (`WHERE`, not `HAVING` — think
   about why)
4. What is the total freight revenue across all orders? (**Expected: R$2,251,909.54**)
5. Find the states where more than 3,000 customers placed orders. (`HAVING`, straight from
   Wednesday)

Work through them together and compare queries before you run the check cell — the
discussion about *why* a query is shaped the way it is matters more than the number it
returns.

## Session Summary

| Pattern | What it does | Example |
|---|---|---|
| `GROUP BY` a small-cardinality column | builds a distribution — one row per distinct value | `GROUP BY review_score` |
| `ORDER BY` the grouped column | reads the *shape* in natural order, not by size | `ORDER BY review_score` |
| Scalar subquery in `SELECT` | supplies a whole-table total to every group | `(SELECT COUNT(*) FROM order_reviews)` |
| `* 100.0` (not `* 100`) | forces REAL division so decimals survive | `COUNT(*) * 100.0 / total` |
| `ROUND(x, 1)` | keeps a percentage column readable | `ROUND(..., 1) AS percentage` |
| Aggregate with no `GROUP BY` | treats the whole table as one group, returns one row | `SELECT AVG(review_score) FROM order_reviews` |
| `HAVING` on the distribution | keeps only buckets above a size threshold | `HAVING COUNT(*) > 10000` |

**Today's numbers worth remembering:** 99,224 reviews; distribution 11.5% / 3.2% / 8.2% /
19.3% / 57.8% across 1→5 stars; overall average **4.09**.

**The analytical lesson:** an average is a summary, and every summary throws information
away. Always ask to see the distribution before you trust a single number — 57.8% 5-star
with an average of 4.09 only makes sense once you can see the 11,424 one-star reviews
sitting at the other end.

---
**Coming up Wednesday (Week 3)**: `JOIN`. Today we could not answer "*which customer
state has the highest average review score?*" — the scores live in `order_reviews` and
the states live in `customers`, and nothing we know yet can put them in the same query.
Next week we connect tables on their shared keys and that question becomes a five-line
query.